# Caso Pratico EMFI 1 – Parte B
**Autori:** Leonardo Pratelli, Sara Albotica – Università di Pisa  
**Docente:** Prof. Fulvio Corsi

---

## Parte B – Stima dei Beta, Frontiera Efficiente, CAPM

Questo notebook copre i **Punti 8 e 9** della traccia:

- **Punto 8** → Stima OLS di alpha e beta per tutti i 10 titoli rispetto all'S&P 500. Verifica della significatività statistica di entrambi i coefficienti.
- **Punto 9** → Ricostruzione della matrice varianza-covarianza tramite il **Single Index Model (SIM)** e dei rendimenti attesi tramite il **CAPM**. Costruzione della frontiera efficiente e confronto con quella empirica della Parte A.

Ogni blocco ha la stessa struttura di `notebook_A`: cella markdown con teoria e formule, seguita dal codice commentato.

---
## Blocco 0 – Import e configurazione

In [ ]:
# =============================================================================
# IMPORT
# Stesse librerie della Parte A + scipy.stats per i test statistici OLS
# matplotlib.patches: usato per costruire leggende personalizzate nei grafici
# =============================================================================
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11

# =============================================================================
# CONFIGURAZIONE (identica alla Parte A per garantire coerenza)
# =============================================================================
# Stessa struttura 3-3-3-1 e stessi parametri della Parte A.
# I 5 titoli selezionati (SELECTED) vengono usati al Punto 9 per confrontare
# la frontiera SIM/CAPM con quella empirica della Parte A.

TICKERS = {
    'Tech':       ['AAPL', 'MSFT', 'NVDA'],
    'Healthcare': ['JNJ',  'PFE',  'MRK'],
    'Energy':     ['XOM',  'CVX',  'BP'],
    'Index':      ['^GSPC'],
}
ALL_TICKERS = [t for group in TICKERS.values() for t in group]

# 5 titoli scelti nella Parte A per il confronto delle frontiere
SELECTED = ['NVDA', 'MSFT', 'JNJ', 'MRK', 'XOM']

# Parametri economici: identici alla Parte A
rf_annual = 0.02        # tasso risk-free annuo (proxy BOT 3 mesi 2015-2025)
ann       = 12          # fattore di annualizzazione
rf        = rf_annual / ann    # tasso mensile: 0.1667%

START = '2015-01-01'
END   = '2025-01-01'

# Palette colori settori (identica alla Parte A)
palette = {'Tech': 'steelblue', 'Healthcare': 'seagreen',
           'Energy': 'tomato', 'Index': 'black'}
sector_map = {}
for sector, tickers in TICKERS.items():
    for t in tickers:
        sector_map['SP500' if t == '^GSPC' else t] = sector

print('Titoli:', ALL_TICKERS)
print(f'rf = {rf_annual*100:.1f}% annuo | Periodo: {START} → {END}')

---
## Blocco 1 – Download dati e rendimenti

Notebook autonomo: scarica i dati da zero, identicamente alla Parte A.  
Così ogni notebook è indipendente e riproducibile senza dipendere dall'esecuzione degli altri.

In [ ]:
# Download prezzi mensili adjusted (total return, dividendi inclusi)
# auto_adjust=True → prezzi già rettificati per dividendi e split
# resample('ME').last() → prezzo di chiusura dell'ultimo giorno del mese
raw    = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True, progress=True)
prices = raw['Close'].resample('ME').last()
prices.columns = [c if c != '^GSPC' else 'SP500' for c in prices.columns]
prices.dropna(how='all', inplace=True)

# Rendimenti logaritmici mensili: r_t = ln(P_t / P_{t-1})
returns = np.log(prices / prices.shift(1)).dropna()

# Serie del mercato (proxy: SP500) usata come regressore in tutte le OLS
r_m      = returns['SP500']
mu_m     = r_m.mean()              # rendimento medio mensile del mercato
sigma2_m = np.var(r_m, ddof=1)     # varianza mensile del mercato (ddof=1)
RISKY    = list(returns.columns)   # tutti i 10 titoli (incluso SP500)

print(f'Osservazioni: {len(returns)} mesi  ({returns.index[0].date()} → {returns.index[-1].date()})')
print(f'μ mercato mensile: {mu_m*100:.3f}%  |  σ² mercato mensile: {sigma2_m*10000:.4f}×10⁻⁴')
returns.head(3)

---
## Blocco 2 – Stima OLS dei beta e degli alpha (Punto 8)

### Modello di regressione (Single Index Model)

$$r_{i,t} = \alpha_i + \beta_i \cdot r_{m,t} + \varepsilon_{i,t}$$

- $\beta_i = \frac{\text{Cov}(r_i,\, r_m)}{\text{Var}(r_m)}$ → **rischio sistematico** (non diversificabile)
- $\alpha_i = \bar{r}_i - \hat{\beta}_i\,\bar{r}_m$ → **alpha di Jensen**: rendimento non spiegato dal rischio di mercato
- $\varepsilon_{i,t}$ → componente **idiosincratica** (specifica dell'asset, diversificabile)

### Inferenza statistica (OLS classico)

Con $\hat{\sigma}^2_\varepsilon = \text{RSS}/(T-2)$ e $S_{xx} = \sum_t(r_{m,t}-\bar{r}_m)^2$:

$$\text{SE}(\hat{\beta}_i) = \sqrt{\frac{\hat{\sigma}^2_\varepsilon}{S_{xx}}}, \qquad
  \text{SE}(\hat{\alpha}_i) = \sqrt{\hat{\sigma}^2_\varepsilon\left(\frac{1}{T}+\frac{\bar{r}_m^2}{S_{xx}}\right)}$$

Le statistiche $t = \hat{\cdot}/\text{SE}$ seguono una $t(T-2)$.  
Il **CAPM** prevede $\alpha_i = 0$ in equilibrio: un alpha significativamente diverso da zero è un'anomalia.

In [ ]:
# =============================================================================
# FUNZIONE OLS – implementazione manuale con tutte le statistiche inferenziali
# =============================================================================
def ols_stats(y_series, x_series):
    """
    OLS: y_t = alpha + beta * x_t + eps_t
    Restituisce stime, errori standard, t-stat, p-value e R².
    """
    y, x = np.array(y_series, float), np.array(x_series, float)
    T    = len(y)

    # Stime OLS
    beta  = np.cov(y, x, ddof=1)[0, 1] / np.var(x, ddof=1)
    alpha = y.mean() - beta * x.mean()

    # Residui e varianza residua (ddof = T-2: due parametri stimati)
    eps = y - alpha - beta * x
    s2  = (eps @ eps) / (T - 2)

    # Errori standard
    Sxx  = ((x - x.mean())**2).sum()     # = (T-1) * Var(x, ddof=1)
    se_b = np.sqrt(s2 / Sxx)
    se_a = np.sqrt(s2 * (1/T + x.mean()**2 / Sxx))

    # t-stat e p-value (due code, distribuzione t con T-2 dof)
    t_b, t_a = beta / se_b, alpha / se_a
    p_b = 2 * stats.t.sf(abs(t_b), df=T - 2)
    p_a = 2 * stats.t.sf(abs(t_a), df=T - 2)

    # R² = 1 - RSS/TSS
    R2 = 1 - (eps @ eps) / ((y - y.mean())**2).sum()

    return dict(alpha=alpha, beta=beta, se_a=se_a, se_b=se_b,
                t_a=t_a, t_b=t_b, p_a=p_a, p_b=p_b,
                R2=R2, eps=eps, s2=s2)

# Simboli di significatività standard
def sig_stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

# =============================================================================
# STIMA OLS PER TUTTI I 10 TITOLI
# =============================================================================
ols = {t: ols_stats(returns[t], r_m) for t in RISKY}

# Tabella riassuntiva
rows = []
for t in RISKY:
    o = ols[t]
    rows.append({
        'Ticker':        t,
        'Alpha ann (%)': round(o['alpha'] * ann * 100, 3),
        't(α)':          round(o['t_a'], 3),
        'Sig α':         sig_stars(o['p_a']),
        'Beta':          round(o['beta'], 4),
        't(β)':          round(o['t_b'], 3),
        'Sig β':         sig_stars(o['p_b']),
        'R²':            round(o['R2'], 4),
    })

tab_ols = pd.DataFrame(rows).set_index('Ticker')
print(f'OLS: r_i = alpha + beta * r_m + eps   (T = {len(returns)} mesi)\n')
display(tab_ols)
print('\nLegenda: *** p<0.01  ** p<0.05  * p<0.10')

---
## Blocco 3 – Visualizzazione dei beta e degli alpha (Punto 8)

**Interpretazione del beta:**
- β > 1 → asset **aggressivo**: amplifica i movimenti del mercato
- β = 1 → si muove in linea con il mercato (per verifica: β(SP500) = 1)
- β < 1 → asset **difensivo**: attutisce le variazioni

**Interpretazione dell'alpha di Jensen:**
- α > 0 → outperformance rispetto alle previsioni del CAPM
- α < 0 → underperformance rispetto alle previsioni del CAPM
- α ≈ 0 → rendimento in linea con la compensazione per il rischio sistematico

In [ ]:
betas_all  = [ols[t]['beta']          for t in RISKY]
alphas_ann = [ols[t]['alpha']*ann*100 for t in RISKY]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Grafico sinistra: Beta ────────────────────────────────────────────────────
# Rosso = aggressivo (β>1), Blu = difensivo (β<1)
ax = axes[0]
colors_b = ['tomato' if b > 1 else 'steelblue' for b in betas_all]
bars = ax.bar(RISKY, betas_all, color=colors_b, edgecolor='black', lw=0.5)
ax.axhline(1.0, color='black', lw=1.5, ls='--', label='β = 1 (mercato)')
ax.axhline(0.0, color='grey',  lw=0.8, ls=':')
for bar, b in zip(bars, betas_all):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{b:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_title('Beta dei 10 titoli vs S&P 500', fontsize=12)
ax.set_ylabel('Beta (β)')
ax.tick_params(axis='x', rotation=30)
ax.legend(handles=[mpatches.Patch(color='tomato',    label='β > 1 (aggressivo)'),
                   mpatches.Patch(color='steelblue', label='β ≤ 1 (difensivo)')], fontsize=9)

# ── Grafico destra: Alpha di Jensen annualizzato ──────────────────────────────
# Verde = outperformance (α>0), Salmone = underperformance (α<0)
ax = axes[1]
colors_a = ['limegreen' if a > 0 else 'salmon' for a in alphas_ann]
bars_a   = ax.bar(RISKY, alphas_ann, color=colors_a, edgecolor='black', lw=0.5)
ax.axhline(0, color='black', lw=1)
for bar, a in zip(bars_a, alphas_ann):
    ypos = bar.get_height() + 0.3 if a >= 0 else bar.get_height() - 1.8
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{a:.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_title('Alpha di Jensen annualizzato (%/anno)', fontsize=12)
ax.set_ylabel('Alpha annuo (%)')
ax.tick_params(axis='x', rotation=30)
ax.legend(handles=[mpatches.Patch(color='limegreen', label='α > 0 (outperformance)'),
                   mpatches.Patch(color='salmon',    label='α < 0 (underperformance)')], fontsize=9)

fig.suptitle('Beta e Alpha di Jensen – Stima OLS (2015–2025)', fontsize=13)
fig.tight_layout()
fig.savefig('beta_alpha_B8.png', dpi=150)
plt.show()
print('Salvato: beta_alpha_B8.png')

---
## Blocco 4 – Security Market Line (Punto 8)

La **Security Market Line (SML)** rappresenta graficamente la relazione del CAPM tra rendimento atteso e beta:

$$\mathbb{E}[r_i] = r_f + \beta_i \cdot (\mathbb{E}[r_m] - r_f)$$

È una **retta** nello spazio $(\beta,\, \mu)$ che passa per $(0, r_f)$ e $(1, \mu_m)$.

**Differenza SML vs CML:**
- **CML** (Parte A): asse x = $\sigma$ (std dev totale), valida solo per portafogli efficienti
- **SML** (Parte B): asse x = $\beta$ (rischio sistematico), valida per **qualsiasi** asset

Asset **sopra** la SML hanno alpha > 0 (rendimento reale > CAPM), asset **sotto** hanno alpha < 0.

In [ ]:
betas_arr   = np.array([ols[t]['beta']    for t in RISKY])
mu_emp_all  = np.array([returns[t].mean() for t in RISKY])
mu_capm_all = rf + betas_arr * (mu_m - rf)   # CAPM previsto mensile

# Retta SML: E[r] = rf + beta * (mu_m - rf)
beta_grid = np.linspace(min(betas_arr) - 0.1, max(betas_arr) + 0.1, 200)
mu_sml    = rf + beta_grid * (mu_m - rf)

fig, ax = plt.subplots(figsize=(11, 7))

# SML
ax.plot(beta_grid, mu_sml * ann * 100, 'b-', lw=2,
        label=f'SML: E[r] = {rf_annual*100:.1f}% + β·({mu_m*ann*100:.1f}% − {rf_annual*100:.1f}%)')

# Punto risk-free (beta=0)
ax.scatter(0, rf_annual * 100, color='grey', s=100, marker='o', zorder=5,
           label=f'rf = {rf_annual*100:.1f}%')

# Singoli asset: posizionati con il rendimento reale storico
for i, t in enumerate(RISKY):
    color = palette[sector_map[t]]
    mu_real = mu_emp_all[i] * ann * 100
    mu_pred = mu_capm_all[i] * ann * 100
    ax.scatter(betas_arr[i], mu_real, color=color, s=120, zorder=5,
               edgecolors='black', lw=0.5)
    ax.annotate(t, (betas_arr[i], mu_real),
                textcoords='offset points', xytext=(6, 3), fontsize=9)
    # Freccia verticale che visualizza l'alpha (distanza dalla SML)
    if abs(mu_real - mu_pred) > 1.5:
        ax.annotate('', xy=(betas_arr[i], mu_real),
                    xytext=(betas_arr[i], mu_pred),
                    arrowprops=dict(arrowstyle='->', color='grey', lw=1.2))

# Legenda settori
legend_patches = [mpatches.Patch(color=c, label=s) for s, c in palette.items()]
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + legend_patches, fontsize=8, loc='upper left')

ax.set_xlabel('Beta (β)', fontsize=12)
ax.set_ylabel('Rendimento Atteso Annuo (%)', fontsize=12)
ax.set_title('Security Market Line (SML) – Rendimenti CAPM vs Rendimenti Reali', fontsize=13)
ax.axhline(0, color='grey', lw=0.5, ls=':')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('SML_B8.png', dpi=150)
plt.show()
print('Salvato: SML_B8.png')

---
## Blocco 5 – Matrice varianza-covarianza via Single Index Model (Punto 9)

Il **SIM** semplifica la struttura della covarianza: assume che l'**unica fonte di correlazione** tra i titoli sia il fattore comune di mercato.

$$\Sigma_{\text{SIM}} = \boldsymbol{\beta}\boldsymbol{\beta}' \cdot \sigma^2_m + \mathbf{D}$$

- Diagonale: $\sigma^2_i = \beta_i^2 \sigma^2_m + \sigma^2_{\varepsilon_i}$ (varianza totale = sistematica + idiosincratica)
- Off-diagonale: $\text{Cov}_{\text{SIM}}(i,j) = \beta_i \beta_j \sigma^2_m$ (tutta la co-movimentazione passa dal mercato)

**Risparmio di parametri:** da $N(N+1)/2$ a $2N+1$
- Con N=10: da 55 parametri a 21
- Con N=100: da 5050 parametri a 201

In [ ]:
# ── Parametri SIM per tutti i 10 titoli ───────────────────────────────────────
# betas_arr: vettore N×1 dei beta OLS (già calcolato nel Blocco 4)
# s2eps_arr: varianze dei residui OLS = componente idiosincratica di ogni titolo
# sigma2_m:  varianza mensile del mercato (già calcolata nel Blocco 1)

s2eps_arr = np.array([ols[t]['s2'] for t in RISKY])   # sigma2_eps_i per ogni titolo

# Sigma_SIM = beta*beta' * sigma2_m + diag(sigma2_eps)
# np.outer(betas_arr, betas_arr) → matrice N×N: elemento [i,j] = beta_i * beta_j
# * sigma2_m                     → scala per la varianza del mercato
# + np.diag(s2eps_arr)           → aggiunge le varianze idiosincratiche sulla diagonale
Sigma_SIM = np.outer(betas_arr, betas_arr) * sigma2_m + np.diag(s2eps_arr)

# Verifica: la diagonale di Sigma_SIM deve coincidere con la varianza empirica
# perché Var(r_i) = beta_i^2 * Var(r_m) + Var(eps_i)  [decomposizione varianza]
var_emp = np.array([returns[t].var(ddof=1) for t in RISKY])
var_sim = np.diag(Sigma_SIM)

print('Verifica: Var(SIM) == Var(empirica) – devono coincidere per costruzione:\n')
check = pd.DataFrame({
    'Var SIM (×10⁴)':      (var_sim * 10000).round(4),
    'Var empirica (×10⁴)': (var_emp * 10000).round(4),
    'Differenza (×10⁶)':   ((var_sim - var_emp) * 1e6).round(4),
}, index=RISKY)
display(check)

# ── Confronto matrici di correlazione: SIM vs empirica ────────────────────────
# Normalizziamo Sigma_SIM → matrice di correlazione SIM
D_inv    = np.diag(1 / np.sqrt(var_sim))
Corr_SIM = pd.DataFrame(D_inv @ Sigma_SIM @ D_inv, index=RISKY, columns=RISKY)
Corr_emp = returns.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, corr, title in zip(axes,
                            [Corr_SIM, Corr_emp],
                            ['Correlazione SIM  (solo fattore mercato)',
                             'Correlazione Empirica  (storica)']):
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, square=True,
                linewidths=0.5, ax=ax, annot_kws={'size': 8})
    ax.set_title(title, fontsize=12)
fig.suptitle('Confronto correlazioni: SIM vs Empirica\n'
             'Il SIM appiattisce le correlazioni perché ignora co-movimenti idiosincratici', fontsize=12)
fig.tight_layout()
fig.savefig('correlation_SIM_vs_empirical_B9.png', dpi=150)
plt.show()
print('Salvato: correlation_SIM_vs_empirical_B9.png')

---
## Blocco 6 – Rendimenti attesi CAPM vs empirici (Punto 9)

Il **CAPM** in equilibrio impone $\alpha_i = 0$: il rendimento atteso è interamente spiegato dal rischio sistematico:

$$\mathbb{E}[r_i]_{\text{CAPM}} = r_f + \beta_i\,(\mu_m - r_f)$$

**Confronto con i rendimenti empirici**: la differenza $\mu_i^{\text{emp}} - \mathbb{E}[r_i]^{\text{CAPM}} = \hat{\alpha}_i$  
Nel grafico scatter, se il CAPM fosse esatto tutti i punti starebbero sulla bisettrice $y = x$.

In [ ]:
# Rendimenti attesi CAPM mensili: E[r_i] = rf + beta_i * (mu_m - rf)
mu_capm_arr = rf + betas_arr * (mu_m - rf)     # array (N,) mensile
mu_emp_arr  = np.array([returns[t].mean() for t in RISKY])

# Tabella confronto
comp = pd.DataFrame({
    'Beta':                 betas_arr.round(4),
    'μ CAPM annuo (%)':     (mu_capm_arr * ann * 100).round(2),
    'μ Empirico annuo (%)': (mu_emp_arr  * ann * 100).round(2),
    'Alpha annuo (%)':      ((mu_emp_arr - mu_capm_arr) * ann * 100).round(2),
    'p-value α':            [round(ols[t]['p_a'], 4) for t in RISKY],
    'Significatività':      [sig_stars(ols[t]['p_a']) for t in RISKY],
}, index=RISKY)

print('Confronto μ empirico vs μ CAPM:\n')
display(comp)

# Scatter: mu_empirico vs mu_CAPM (se CAPM fosse perfetto → tutti sulla bisettrice)
fig, ax = plt.subplots(figsize=(8, 8))
all_mu = np.concatenate([mu_capm_arr, mu_emp_arr]) * ann * 100
lim    = (min(all_mu) - 3, max(all_mu) + 3)
ax.plot(lim, lim, 'k--', lw=1.2, label='y = x  (CAPM perfetto: α = 0)')

for i, t in enumerate(RISKY):
    color = palette[sector_map[t]]
    ax.scatter(mu_capm_arr[i]*ann*100, mu_emp_arr[i]*ann*100,
               color=color, s=130, zorder=4, edgecolors='black', lw=0.5)
    ax.annotate(t, (mu_capm_arr[i]*ann*100, mu_emp_arr[i]*ann*100),
                textcoords='offset points', xytext=(5, 3), fontsize=9)

ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Rendimento CAPM annuo (%)'); ax.set_ylabel('Rendimento Empirico annuo (%)')
ax.set_title('CAPM previsto vs Rendimento Storico\n'
             'Punti sopra la bisettrice → α > 0 (outperformance)', fontsize=12)
legend_patches = [mpatches.Patch(color=c, label=s) for s, c in palette.items()]
ax.legend(handles=ax.get_legend_handles_labels()[0] + legend_patches, fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('capm_vs_empirical_B9.png', dpi=150)
plt.show()
print('Salvato: capm_vs_empirical_B9.png')

---
## Blocco 7 – Frontiera efficiente SIM/CAPM vs Empirica (Punto 9)

Si costruisce la frontiera efficiente usando **μ_CAPM** e **Σ_SIM** per i 5 titoli selezionati nella Parte A e la si confronta con la frontiera empirica (μ empirico, Σ empirica).

**Perché le due frontiere differiscono:**
1. **Rendimenti**: μ_CAPM ≠ μ_empirico → il CAPM azzera gli alpha, i rendimenti storici no
2. **Covarianze**: Σ_SIM ≠ Σ_empirica → il SIM semplifica la struttura (solo fattore mercato)

La formula analitica è la stessa della Parte A: $\sigma^2(\mu_p) = (A\mu_p^2 - 2B\mu_p + C)/D$

In [ ]:
# ── Estrai parametri per i 5 titoli selezionati ───────────────────────────────
sel_idx  = [RISKY.index(t) for t in SELECTED]   # posizioni nell'array RISKY

# Parte A: mu e Sigma empirici
mu5_emp  = np.array([returns[t].mean() for t in SELECTED])   # (5,) mensili
Sig5_emp = returns[SELECTED].cov().values                      # (5×5) empirica

# Parte B: mu CAPM e Sigma SIM
mu5_capm = mu_capm_arr[sel_idx]                               # (5,) mensili CAPM
Sig5_SIM = Sigma_SIM[np.ix_(sel_idx, sel_idx)]               # (5×5) SIM

# ── Funzione frontiera analitica (scalari di Markowitz) ───────────────────────
def markowitz_frontier(mu_vec, Sig_mat, n_pts=300):
    """
    Frontiera efficiente analitica data μ e Σ.
    Usa le formule dei scalari A, B, C, D della Parte A.
    Ritorna (sigma_annua %, mu_annua %, mu_gmv mensile).
    """
    iota = np.ones(len(mu_vec))
    Sinv = np.linalg.inv(Sig_mat)
    A_mk = float(iota @ Sinv @ iota)
    B_mk = float(iota @ Sinv @ mu_vec)
    C_mk = float(mu_vec @ Sinv @ mu_vec)
    D_mk = A_mk * C_mk - B_mk**2
    mu_gmv  = B_mk / A_mk
    mu_grid = np.linspace(mu_gmv, mu_vec.max() * 1.6, n_pts)
    sig_grid = np.sqrt((A_mk * mu_grid**2 - 2*B_mk*mu_grid + C_mk) / D_mk)
    return sig_grid * np.sqrt(ann) * 100, mu_grid * ann * 100, mu_gmv

# Frontiere
sig_emp_f, mu_emp_f, mu_gmv_emp = markowitz_frontier(mu5_emp,  Sig5_emp)
sig_sim_f, mu_sim_f, mu_gmv_sim = markowitz_frontier(mu5_capm, Sig5_SIM)

# ── Grafico confronto ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(sig_emp_f, mu_emp_f, 'b-',  lw=2.5,
        label='Frontiera empirica  (μ empirico, Σ empirica) – Parte A')
ax.plot(sig_sim_f, mu_sim_f, 'r--', lw=2.5,
        label='Frontiera SIM/CAPM  (μ_CAPM, Σ_SIM) – Parte B')

# GMV delle due frontiere
ax.scatter(returns[SELECTED].std().values @ np.ones(5) * 0,  # placeholder (non usato)
           0, alpha=0)   # dummy per evitare conflitti zorder

# Singoli asset: cerchio = mu empirico, croce = mu CAPM
for i, t in enumerate(SELECTED):
    sig_a = returns[t].std()  * np.sqrt(ann) * 100
    mu_a  = returns[t].mean() * ann * 100
    mu_c  = mu5_capm[i] * ann * 100
    ax.scatter(sig_a, mu_a, s=110, marker='o', color='steelblue',
               zorder=5, edgecolors='black', lw=0.5)
    ax.scatter(sig_a, mu_c, s=110, marker='x', color='red',
               zorder=5, linewidths=2)
    ax.annotate(t, (sig_a, mu_a), textcoords='offset points', xytext=(5, 3), fontsize=9)

extra_legend = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue',
           markersize=9, markeredgecolor='black', label='Asset: μ empirico'),
    Line2D([0],[0], marker='x', color='red', markersize=9,
           lw=0, markeredgewidth=2, label='Asset: μ CAPM'),
]
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles + extra_legend, fontsize=9)

ax.set_xlabel('Deviazione Standard Annua (%)', fontsize=12)
ax.set_ylabel('Rendimento Atteso Annuo (%)', fontsize=12)
ax.set_title('Frontiera Efficiente: Empirica (Parte A) vs SIM/CAPM (Parte B)\n'
             '5 titoli selezionati – short selling ammesso', fontsize=13)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('frontier_SIM_vs_empirical_B9.png', dpi=150)
plt.show()
print('Salvato: frontier_SIM_vs_empirical_B9.png')

print('\nCommento:')
print('- La frontiera SIM/CAPM differisce da quella empirica principalmente')
print('  per i rendimenti attesi: μ_CAPM=rf+β(μ_m-rf) può differire molto')
print('  dai rendimenti storici quando gli alpha di Jensen sono grandi.')
print('- La struttura delle covarianze SIM è più parsimoniosa (2N+1 param.)')
print('  ma può perdere informazione sulle correlazioni idiosincratiche.')